# Validation of all models

In [14]:
import pandas as pd

models_results = pd.DataFrame([], columns=["Precision", "Recall", "F1-Score", "mAP50", "mAP50-95", "Loss", "Fitness", "Inference Speed (ms)"])
multiclass_results = pd.DataFrame([], columns=["Precision", "Recall", "F1-Score", "mAP50", "mAP50-95", "Loss", "Fitness", "Inference Speed (ms)"])

In [2]:
uniclass_paths = [
    "Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nBlackRot/YOLO12n_BlackRot_AllImages_trained.pt",
    "Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nEsca/YOLO12n_Esca_AllImages_trained.pt",
    "Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nLeafBlight/YOLO12n_LeafBlight_AllImages_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_2/YOLOv12nBlackRot/YOLO12n_BlackRot_WithoutOutliers_1_2_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_2/YOLOv12nEsca/YOLO12n_Esca_WithoutOutliers_1_2_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_2/YOLOv12nLeafBlight/YOLO12n_LeafBlight_WithoutOutliers_1_2_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nBlackRot/YOLO12n_BlackRot_WithoutOutliers_1_5_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nEsca/YOLO12n_Esca_WithoutOutliers_1_5_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nLeafBlight/YOLO12n_LeafBlight_WithoutOutliers_1_5_trained.pt"
]

multiclass_paths = [
    "Yolov12n_ObjectDetection_Classification/AllData/YOLOv12n3Classes/YOLO12n_3Classes_AllImages_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_2/YOLOv12n3Classes/YOLO12n_3Classes_WithoutOutliers_1_2_trained.pt",
    "Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12n3Classes/YOLO12n_3Classes_WithoutOutliers_1_5_trained.pt"
] 

In [3]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Define paths

validation_directory = "Yolov12n_ObjectDetection_Classification/ValidationFolder"

sub_folders = [
    "AllData",
    "WithoutOutliers_1_2",
    "WithoutOutliers_1_5"
]

labels = [
    "3Classes",
    "BlackRot",
    "Esca",
    "LeafBlight"
] 

label_class = [
    "BlackRot",
    "Esca",
    "LeafBlight"
] 

if not os.path.isdir(validation_directory):

    os.makedirs(validation_directory, exist_ok=True)    

    for sub_folder in sub_folders:
        os.makedirs(validation_directory + "/" + sub_folder, exist_ok=True)    

        for label in labels:
            os.makedirs(validation_directory + "/" + sub_folder + "/" + label, exist_ok=True)    

#### Uniclass

In [14]:
import os
import shutil
from ultralytics import YOLO
import yaml

for uniclass_path in uniclass_paths:

    subfolder_path = "" 
    label_path = ""

    # Verify Folder and Label
    for f in sub_folders:
        if f in uniclass_path:
            subfolder_path = f
            break

    for l in labels:
        if l in uniclass_path:
            label_path = l
            break

    print("Subfolder: " + subfolder_path)
    print("Label: " + label_path)

    if not os.path.isfile(uniclass_path):
        print("O ficheiro " + uniclass_path + " não existe")

    config_path = uniclass_path.replace(uniclass_path.split("/")[-1], "") + "config.yaml"

    # YAML
    with open(config_path) as file:
        yaml_data = yaml.safe_load(file)

    yaml_data['path'] = uniclass_path.replace(uniclass_path.split("/")[-1], "") + "/Dataset"

    with open("config.yaml", "w") as file:
        yaml.dump(yaml_data, file)


    model = YOLO(uniclass_path)

    for conf_value in [0.3]: 

        results = model.val(data="config.yaml", iou=0.7, conf=conf_value, batch=128, plots=True, name=subfolder_path + "_" + label_path + "_" + "/Validation")

        f1score = 2 * (results.results_dict['metrics/precision(B)'] * results.results_dict['metrics/recall(B)']) / (results.results_dict['metrics/precision(B)'] + results.results_dict['metrics/recall(B)'])
        
        models_results.loc[subfolder_path + "_" + label_path + "_" + str(conf_value)] = [
            results.results_dict['metrics/precision(B)'],
            results.results_dict['metrics/recall(B)'],
            f1score,
            results.results_dict['metrics/mAP50(B)'],
            results.results_dict['metrics/mAP50-95(B)'],
            results.speed['loss'],
            results.results_dict['fitness'],
            results.speed['inference'],
        ]
        

Subfolder: AllData
Label: BlackRot
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 724.3±159.0 MB/s, size: 16.6 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nBlackRot/Dataset/labels/val/BlackRot.cache... 268 images, 0 backgrounds, 0 corrupt: 100%|██████████| 268/268 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:04<00:00,  1.55s/it]


                   all        268       2132      0.759      0.773      0.782      0.433
Speed: 3.2ms preprocess, 1.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/AllData_BlackRot_/Validation9
Subfolder: AllData
Label: Esca
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 818.5±134.2 MB/s, size: 19.9 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nEsca/Dataset/labels/val/Esca.cache... 240 images, 0 backgrounds, 0 corrupt: 100%|██████████| 240/240 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:04<00:00,  2.06s/it]


                   all        240       2959      0.785       0.83      0.845      0.499
Speed: 1.9ms preprocess, 2.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/AllData_Esca_/Validation3
Subfolder: AllData
Label: LeafBlight
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 808.3±190.8 MB/s, size: 18.6 KB)


...

                   all        266       2126      0.743      0.775      0.781      0.426
Speed: 3.2ms preprocess, 1.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_BlackRot_/Validation5
Subfolder: WithoutOutliers_1_5
Label: Esca
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 793.6±176.6 MB/s, size: 19.2 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nEsca/Dataset/labels/val/Esca.cache... 240 images, 0 backgrounds, 0 corrupt: 100%|██████████| 240/240 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:04<00:00,  2.01s/it]


                   all        240       2672      0.769       0.83      0.831      0.475
Speed: 1.8ms preprocess, 2.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_Esca_/Validation3
Subfolder: WithoutOutliers_1_5
Label: LeafBlight
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 761.1±183.0 MB/s, size: 17.7 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nLeafBlight/Dataset/labels/val/LeafBlight.cache... 215 images, 0 backgrounds, 0 corrupt: 100%|██████████| 215/215 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:03<00:00,  1.82s/it]


                   all        215       2935      0.757      0.768      0.789      0.404
Speed: 0.1ms preprocess, 3.9ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_LeafBlight_/Validation3


General

In [5]:
models_results[models_results.index.str.contains("0.3")] 

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData_BlackRot_0.3,0.778045,0.766194,0.772074,0.779592,0.431903,0.000175,0.466672,2.535955
AllData_Esca_0.3,0.820294,0.811423,0.815834,0.843461,0.497900,0.000177,0.532456,2.106191
AllData_LeafBlight_0.3,0.777742,0.779981,0.778860,0.799488,0.418670,0.000134,0.456751,3.978295
WithoutOutliers_1_2_BlackRot_0.3,0.761954,0.711996,0.736129,0.743030,0.423624,0.000149,0.455564,2.277953
WithoutOutliers_1_2_Esca_0.3,0.788743,0.790766,0.789753,0.810659,0.453659,0.000130,0.489359,2.042398
WithoutOutliers_1_2_LeafBlight_0.3,0.758419,0.753282,0.755842,0.774822,0.395347,0.000119,0.433294,2.100722
WithoutOutliers_1_5_BlackRot_0.3,0.768972,0.753057,0.760932,0.776221,0.424152,0.000147,0.459359,2.257108
WithoutOutliers_1_5_Esca_0.3,0.784335,0.820734,0.802121,0.829757,0.474360,0.000129,0.509900,2.021067
WithoutOutliers_1_5_LeafBlight_0.3,0.770898,0.763543,0.767203,0.789633,0.403663,0.000140,0.442260,3.953093


BlackRot

In [6]:
models_results[models_results.index.str.contains("BlackRot_0.3")] 

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData_BlackRot_0.3,0.778045,0.766194,0.772074,0.779592,0.431903,0.000175,0.466672,2.535955
WithoutOutliers_1_2_BlackRot_0.3,0.761954,0.711996,0.736129,0.743030,0.423624,0.000149,0.455564,2.277953
WithoutOutliers_1_5_BlackRot_0.3,0.768972,0.753057,0.760932,0.776221,0.424152,0.000147,0.459359,2.257108


Esca

In [7]:
models_results[models_results.index.str.contains("Esca_0.3")] 

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData_Esca_0.3,0.820294,0.811423,0.815834,0.843461,0.497900,0.000177,0.532456,2.106191
WithoutOutliers_1_2_Esca_0.3,0.788743,0.790766,0.789753,0.810659,0.453659,0.000130,0.489359,2.042398
WithoutOutliers_1_5_Esca_0.3,0.784335,0.820734,0.802121,0.829757,0.474360,0.000129,0.509900,2.021067


Leaf Blight

In [8]:
models_results[models_results.index.str.contains("LeafBlight_0.3")] 

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData_LeafBlight_0.3,0.777742,0.779981,0.778860,0.799488,0.418670,0.000134,0.456751,3.978295
WithoutOutliers_1_2_LeafBlight_0.3,0.758419,0.753282,0.755842,0.774822,0.395347,0.000119,0.433294,2.100722
WithoutOutliers_1_5_LeafBlight_0.3,0.770898,0.763543,0.767203,0.789633,0.403663,0.000140,0.442260,3.953093


In [30]:
import pandas as pd

models_results = pd.DataFrame([], columns=["Precision", "Recall", "F1-Score", "mAP50", "mAP50-95", "Loss", "Fitness", "Inference Speed (ms)"])
multiclass_results = pd.DataFrame([], columns=["Precision", "Recall", "F1-Score", "mAP50", "mAP50-95", "Loss", "Fitness", "Inference Speed (ms)"])

### Multiclass

In [31]:
import os
import shutil
from ultralytics import YOLO
import yaml

for multiclass_path in multiclass_paths:

    subfolder_path = "" 
    label_path = ""

    

    for l in label_path:
        if l in multiclass_path:
            label_path = l
            break

    for f in sub_folders:
        if f in multiclass_path:
            subfolder_path = f
            break


    config_path = multiclass_path.replace(multiclass_path.split("/")[-1], "") + "config.yaml"

    # YAML
    with open(config_path) as file:
        yaml_data = yaml.safe_load(file)

    yaml_data['path'] = multiclass_path.replace(multiclass_path.split("/")[-1], "") + "/Dataset"

    with open("config.yaml", "w") as file:
        yaml.dump(yaml_data, file)


    model = YOLO(multiclass_path)

    for conf_value in [0.25]: 

        results = model.val(data="config.yaml", split="test", iou=0.20, conf=conf_value, batch=128, plots=True, name=subfolder_path + "_" + label_path + "_" + "/Validation")

        f1score = 2 * (results.results_dict['metrics/precision(B)'] * results.results_dict['metrics/recall(B)']) / (results.results_dict['metrics/precision(B)'] + results.results_dict['metrics/recall(B)'])
        
        multiclass_results.loc[subfolder_path + "_" + label_path + "_" + str(conf_value)] = [
            results.results_dict['metrics/precision(B)'],
            results.results_dict['metrics/recall(B)'],
            f1score,
            results.results_dict['metrics/mAP50(B)'],
            results.results_dict['metrics/mAP50-95(B)'],
            results.speed['loss'],
            results.results_dict['fitness'],
            results.speed['inference'],
        ]

        for i in range(len(label_class)):
            r = results.box.class_result(i)
            multiclass_results.loc[subfolder_path + "_" + label_path + "_" + label_class[i] + "_" + str(conf_value)] = [
                r[0],
                r[1],
                f1score,
                r[2],
                r[3],
                results.speed['loss'],
                results.results_dict['fitness'],
                results.speed['inference'],
            ]

Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,557,313 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 726.9±130.2 MB/s, size: 18.2 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/AllData/YOLOv12n3Classes/Dataset/labels/test/BlackRot.cache... 724 images, 0 backgrounds, 0 corrupt: 100%|██████████| 724/724 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:09<00:00,  1.54s/it]


                   all        724       8658      0.735      0.787      0.784      0.433
             Black_rot        268       2137      0.749      0.775      0.779      0.436
                  Esca        240       3068      0.701      0.812      0.779      0.448
            LeafBlight        216       3453      0.756      0.776      0.795      0.414
Speed: 1.9ms preprocess, 1.9ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs/detect/AllData__/Validation14
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,557,313 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 658.8±198.9 MB/s, size: 17.0 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_2/YOLOv12n3Classes/Dataset/labels/test/BlackRot.cache... 722 images, 0 backgrounds, 0 corrupt: 100%|██████████| 722/722 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:08<00:00,  1.44s/it]


                   all        722       7813      0.739      0.766      0.761      0.412
             Black_rot        266       2045      0.727       0.79      0.771      0.423
                  Esca        240       2634      0.751      0.764      0.754      0.429
            LeafBlight        216       3134      0.739      0.743      0.758      0.385
Speed: 1.9ms preprocess, 1.9ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_2__/Validation14
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 690.2±129.6 MB/s, size: 17.4 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12n3Classes/Dataset/labels/test/BlackRot.cache... 722 images, 0 backgrounds, 0 corrupt: 100%|██████████| 722/722 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:08<00:00,  1.43s/it]


                   all        722       7988      0.735      0.761      0.765      0.415
             Black_rot        266       2027      0.739      0.765       0.78      0.426
                  Esca        240       2769      0.728      0.753      0.744      0.426
            LeafBlight        216       3192      0.739      0.765      0.772      0.393
Speed: 1.8ms preprocess, 1.9ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5__/Validation14


In [32]:
import os
import shutil
from ultralytics import YOLO
import yaml

for uniclass_path in uniclass_paths:

    subfolder_path = "" 
    label_path = ""

    # Verify Folder and Label
    for f in sub_folders:
        if f in uniclass_path:
            subfolder_path = f
            break

    for l in labels:
        if l in uniclass_path:
            label_path = l
            break

    print("Subfolder: " + subfolder_path)
    print("Label: " + label_path)

    if not os.path.isfile(uniclass_path):
        print("O ficheiro " + uniclass_path + " não existe")

    config_path = uniclass_path.replace(uniclass_path.split("/")[-1], "") + "config.yaml"

    # YAML
    with open(config_path) as file:
        yaml_data = yaml.safe_load(file)

    yaml_data['path'] = uniclass_path.replace(uniclass_path.split("/")[-1], "") + "/Dataset"

    with open("config.yaml", "w") as file:
        yaml.dump(yaml_data, file)


    model = YOLO(uniclass_path)

    for conf_value in [0.25]: 

        results = model.val(data="config.yaml", split="test", iou=0.20, conf=conf_value, batch=128, plots=True, name=subfolder_path + "_" + label_path + "_" + "/Validation")

        f1score = 2 * (results.results_dict['metrics/precision(B)'] * results.results_dict['metrics/recall(B)']) / (results.results_dict['metrics/precision(B)'] + results.results_dict['metrics/recall(B)'])
        
        models_results.loc[subfolder_path + "_" + label_path + "_" + str(conf_value)] = [
            results.results_dict['metrics/precision(B)'],
            results.results_dict['metrics/recall(B)'],
            f1score,
            results.results_dict['metrics/mAP50(B)'],
            results.results_dict['metrics/mAP50-95(B)'],
            results.speed['loss'],
            results.results_dict['fitness'],
            results.speed['inference'],
        ]
        

Subfolder: AllData
Label: BlackRot
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 677.0±194.7 MB/s, size: 16.2 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nBlackRot/Dataset/labels/test/BlackRot.cache... 268 images, 0 backgrounds, 0 corrupt: 100%|██████████| 268/268 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:04<00:00,  1.45s/it]


                   all        268       2137      0.762      0.794      0.782      0.427
Speed: 1.6ms preprocess, 3.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/AllData_BlackRot_/Validation17
Subfolder: AllData
Label: Esca
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 702.8±138.1 MB/s, size: 18.1 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/AllData/YOLOv12nEsca/Dataset/labels/test/Esca.cache... 240 images, 0 backgrounds, 0 corrupt: 100%|██████████| 240/240 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:03<00:00,  1.93s/it]


                   all        240       3068       0.81      0.806      0.838      0.486
Speed: 1.8ms preprocess, 2.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/AllData_Esca_/Validation11
Subfolder: AllData
Label: LeafBlight
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 714.2±138.0 MB/s, size: 18.3 KB)


...

                   all        266       2027      0.751      0.777      0.791      0.432
Speed: 3.1ms preprocess, 2.0ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_BlackRot_/Validation13
Subfolder: WithoutOutliers_1_5
Label: Esca
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 662.5±224.5 MB/s, size: 17.2 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nEsca/Dataset/labels/test/Esca.cache... 240 images, 0 backgrounds, 0 corrupt: 100%|██████████| 240/240 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:04<00:00,  2.03s/it]


                   all        240       2769      0.778      0.808      0.821       0.47
Speed: 1.8ms preprocess, 2.1ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_Esca_/Validation11
Subfolder: WithoutOutliers_1_5
Label: LeafBlight
Ultralytics 8.3.171 🚀 Python-3.8.3 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 711.0±191.8 MB/s, size: 18.1 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreino_finalYOLOOnly/Yolov12n_ObjectDetection_Classification/Outliers_1_5/YOLOv12nLeafBlight/Dataset/labels/test/LeafBlight.cache... 216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 216/216 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:03<00:00,  1.90s/it]


                   all        216       3192      0.745      0.755      0.771      0.392
Speed: 2.0ms preprocess, 1.9ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/WithoutOutliers_1_5_LeafBlight_/Validation11


In [35]:
multiclass_results

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData__0.25,0.735486,0.787468,0.760590,0.784104,0.432652,0.000126,0.467798,1.890346
AllData__BlackRot_0.25,0.749321,0.774918,0.760590,0.778901,0.436094,0.000126,0.467798,1.890346
AllData__Esca_0.25,0.701493,0.811930,0.760590,0.778789,0.447913,0.000126,0.467798,1.890346
AllData__LeafBlight_0.25,0.755643,0.775557,0.760590,0.794624,0.413949,0.000126,0.467798,1.890346
WithoutOutliers_1_2__0.25,0.739090,0.765596,0.752110,0.761117,0.412388,0.000135,0.447261,1.887175
WithoutOutliers_1_2__BlackRot_0.25,0.727150,0.789731,0.752110,0.770854,0.423206,0.000135,0.447261,1.887175
WithoutOutliers_1_2__Esca_0.25,0.750839,0.764237,0.752110,0.754441,0.428868,0.000135,0.447261,1.887175
WithoutOutliers_1_2__LeafBlight_0.25,0.739282,0.742821,0.752110,0.758055,0.385088,0.000135,0.447261,1.887175
WithoutOutliers_1_5__0.25,0.735462,0.760778,0.747906,0.765406,0.414796,0.000135,0.449857,1.873092
WithoutOutliers_1_5__BlackRot_0.25,0.739151,0.764677,0.747906,0.779997,0.425542,0.000135,0.449857,1.873092


In [36]:
models_results

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
AllData_BlackRot_0.25,0.761602,0.794104,0.777513,0.781508,0.427333,0.000199,0.462750,3.520401
AllData_Esca_0.25,0.810315,0.806389,0.808347,0.838075,0.485656,0.000145,0.520898,2.061765
AllData_LeafBlight_0.25,0.756699,0.752968,0.754829,0.781780,0.403111,0.000118,0.440978,1.870385
WithoutOutliers_1_2_BlackRot_0.25,0.753285,0.756968,0.755122,0.764502,0.425510,0.000216,0.459409,2.027890
WithoutOutliers_1_2_Esca_0.25,0.786987,0.821944,0.804085,0.830155,0.466639,0.000149,0.502990,2.049469
WithoutOutliers_1_2_LeafBlight_0.25,0.732716,0.757498,0.744901,0.763879,0.385938,0.000145,0.423732,1.869408
WithoutOutliers_1_5_BlackRot_0.25,0.751313,0.776517,0.763707,0.790696,0.431974,0.000157,0.467846,1.990966
WithoutOutliers_1_5_Esca_0.25,0.778474,0.808417,0.793163,0.820798,0.470325,0.000247,0.505372,2.071839
WithoutOutliers_1_5_LeafBlight_0.25,0.744668,0.754699,0.749650,0.770722,0.391608,0.000145,0.429520,1.874145


In [ ]:
#multiclass_results.to_csv("multiclass_results.csv")